# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [1]:
# Runtime → Change runtime type → T4 GPU
# Kernel chạy trên VM Colab (kể cả qua extension VS Code) nên phải clone repo về VM.
import os
REPO = "https://github.com/xuanlocc2/K4-Track-4-Day10-VisionLanguageModel.git"
if not os.path.exists("/content/lab"):
    !git clone -q {REPO} /content/lab
%cd /content/lab
!git pull -q
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

/content/lab
GPU 0: Tesla T4 (UUID: GPU-43a071a7-4ed1-32a6-2b55-2530cefb36df)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 116.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.7 MB/s eta 0:00:00:00:0100:01


In [2]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from getpass import getpass
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except Exception as e:  # extension VS Code có thể không hỗ trợ Secrets
    print('Không đọc được Colab Secrets:', e)
    os.environ['KAGGLE_USERNAME'] = input('KAGGLE_USERNAME: ')
    os.environ['KAGGLE_KEY'] = getpass('KAGGLE_KEY: ')

# Drive để sao lưu adapter/evidence (session free có thể bị ngắt)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP = '/content/drive/MyDrive/lab_day10'
except Exception as e:
    print('Không mount được Drive:', e)
    BACKUP = '/content/backup'
os.makedirs(BACKUP, exist_ok=True)
print('BACKUP =', BACKUP)

Mounted at /content/drive
BACKUP = /content/drive/MyDrive/lab_day10


## 1. Dữ liệu: tải và chia split

In [3]:
!python scripts/prepare_data.py --kaggle

Dataset URL: https://www.kaggle.com/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes
License(s): MIT
100% 185M/185M [00:00<00:00, 261MB/s] 

dataset_version : d10-21e4070301
  train  1422 ảnh  boxes={'pothole': 1074, 'crack': 1750, 'manhole': 691}
  val     276 ảnh  boxes={'pothole': 66, 'crack': 470, 'manhole': 103}
  test    311 ảnh  boxes={'pothole': 121, 'crack': 299, 'manhole': 163}


In [4]:
!python -m pytest -q   # kỳ vọng: 69 passed (có data) hoặc 68 passed, 1 skipped

...........................s.........................................    [100%]
68 passed, 1 skipped in 0.51s


In [ ]:
# Mọi logic nằm trong lab.py (mã của mình); notebook chỉ gọi hàm.
import importlib, json, os, re, time, yaml, torch
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import lab; importlib.reload(lab)
from lab import *
CFG = {n: yaml.safe_load(open(f'configs/{n}.yaml', encoding='utf-8')) for n in ('e0','e1','e2','e3','e4','e5')}
S = {s: load_split(s) for s in ('train', 'val', 'test')}
for s, es in S.items():
    print(f"{s:5s} {len(es):5d} ảnh  {dict(Counter(o['label'] for e in es for o in e['objects']))}  "
          f"sessions={len({e['session'] for e in es})}  box/ảnh max={max(len(e['objects']) for e in es)}")
sess = {s: {e['session'] for e in es} for s, es in S.items()}
assert not (sess['train'] & sess['val'] or sess['train'] & sess['test'] or sess['val'] & sess['test'])
print('Không session nào nằm ở hai split ✓')

def show(paths, cols=3, w=6):
    rows = (len(paths) + cols - 1) // cols
    fig, ax = plt.subplots(rows, cols, figsize=(w * cols, w * 0.6 * rows), squeeze=False)
    for a in ax.flat: a.axis('off')
    for a, pth in zip(ax.flat, paths): a.imshow(Image.open(pth)); a.set_title(pth.split('/')[-1], fontsize=8)
    plt.tight_layout(); plt.show()

paths = []
for i, e in enumerate(S['train'][::237][:6]):
    draw(e, '[]', f'/content/viz/gt_{i}.jpg', e['session']); paths.append(f'/content/viz/gt_{i}.jpg')
show(paths)

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
e = next(e for e in S['train'] if len(e['objects']) >= 3)
ans = objects_to_answer(e['objects'])
print(ans)
# Roundtrip: JSON 0–1000 → pixel, so với box YOLO gốc (pixel)
from scripts.score import _gt_boxes
px = lambda b: (b[0]*.64, b[1]*.36, b[2]*.64, b[3]*.36)
err = max(min(max(abs(a - g) for a, g in zip(px(p['bbox']), px(gt['bbox']))) for gt in _gt_boxes(e))
          for p in parse_output(ans))
print(f'sai số roundtrip lớn nhất = {err:.2f} px (≤ ~1px do làm tròn)')
draw(e, ans, '/content/viz/roundtrip.jpg', 'roundtrip: do phai trung xanh'); show(['/content/viz/roundtrip.jpg'], cols=1, w=9)
print(PROMPT)

## 3. E0: zero-shot

In [ ]:
# Đếm tham số từng khối trên meta device (không tốn GPU) — dùng cho Câu 1
from transformers import AutoConfig, Qwen3VLForConditionalGeneration
with torch.device('meta'):
    meta_model = Qwen3VLForConditionalGeneration._from_config(AutoConfig.from_pretrained(MODEL_ID))
names = [n for n, _ in meta_model.named_modules()]
print({k: f'{v/1e6:.1f}M' for k, v in count_params(meta_model).items()})
print('merger:', sum(p.numel() for n, p in meta_model.named_parameters() if 'visual.merger' in n) / 1e6, 'M',
      ' deepstack:', sum(p.numel() for n, p in meta_model.named_parameters() if 'deepstack_merger_list' in n) / 1e6, 'M')
print('LoRA LLM modules =', sum(bool(re.fullmatch(LLM_RE, n)) for n in names),
      ' LoRA ViT modules =', sum(bool(re.fullmatch(VIT_RE, n)) for n in names))
del meta_model

In [ ]:
# E0 zero-shot trên 311 ảnh test (resume được nếu mất session)
model, processor = load_model()
torch.cuda.reset_peak_memory_stats(); t0 = time.time()
preds = predict(model, processor, S['test'], f'{BACKUP}/E0/pred_test.jsonl', max_new_tokens=CFG['e0']['max_new_tokens'])
E0_INFO = dict(infer_minutes=round((time.time() - t0) / 60, 1), peak_vram_gb=round(torch.cuda.max_memory_allocated() / 2**30, 2))
print(E0_INFO)
for p in preds[:5]: print('---', p['image_id'], '\n', p['raw'][:400])
del model; cleanup()

In [ ]:
EV = {'E0': score(f'{BACKUP}/E0/pred_test.jsonl', 'E0', 'report/evidence/e0.json')}

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(MODEL_ID)
e = S['train'][0]
ex = encode_example(processor, load_image(e), PROMPT, objects_to_answer(e['objects']))
t, h, w = ex['image_grid_thw'][0].tolist(); m = processor.image_processor.merge_size
img_id = processor.tokenizer.convert_tokens_to_ids(processor.image_token)
print('grid_thw =', (t, h, w), '→ token ảnh =', t*h*w // m**2, '| đếm trong input_ids =', (ex['input_ids'] == img_id).sum().item())
keep_ = ex['labels'] != -100
print(f"độ dài chuỗi = {len(ex['input_ids'])}, token tính loss = {keep_.sum().item()} ({keep_.float().mean():.1%})")
print('decode phần có loss:', repr(processor.tokenizer.decode(ex['input_ids'][keep_])))
b = collate([ex, encode_example(processor, load_image(S['train'][1]), PROMPT, objects_to_answer(S['train'][1]['objects']))],
            processor.tokenizer.pad_token_id)
print({k: tuple(v.shape) for k, v in b.items()})

## 5. Setup LoRA / đóng băng cho E1, E2, E3

`apply_setup()` trong `lab.py`: LoRA theo regex `LLM_RE` (252 module) và `VIT_RE` (96 module, chỉ E3);
E2/E3 mở `visual.merger` + `visual.deepstack_merger_list` và đưa về float32. Số module LoRA và tham số huấn luyện được in khi bắt đầu train (ô dưới).

## 6. Vòng huấn luyện

Mỗi ô: train 300 bước → lưu adapter (+ projector.pt) lên Drive → **nạp lại từ đĩa** → suy luận test → chấm. Mất session thì chạy lại ô import ở phần 1 rồi chạy lại ô đó: phần đã xong được bỏ qua.

In [ ]:
R = {}
R['E1'], EV['E1'] = experiment(CFG['e1'], f'{BACKUP}/E1', 'report/evidence/e1.json')

In [ ]:
R['E2'], EV['E2'] = experiment(CFG['e2'], f'{BACKUP}/E2', 'report/evidence/e2.json')

In [ ]:
R['E3'], EV['E3'] = experiment(CFG['e3'], f'{BACKUP}/E3', 'report/evidence/e3.json')

## 7. Suy luận và chấm E1–E3

In [ ]:
# Bảng tổng hợp + đường loss + ảnh minh hoạ (dùng cho REPORT.md)
import glob
EV = {os.path.basename(f)[:-5].upper(): json.load(open(f)) for f in sorted(glob.glob('report/evidence/*.json'))}
print('| Mã | Tham số huấn luyện | VRAM đỉnh (GB) | Phút | valid | F1 | CI95 | FP/ảnh | F1 pothole | F1 crack | F1 manhole |')
for k, d in EV.items():
    mt, pc = d.get('meta', {}), d['per_class']
    print(f"| {k} | {mt.get('trainable_params', 0):,} | {mt.get('peak_vram_gb', '')} | {mt.get('train_minutes', '')} | "
          f"{d['parse_valid_rate']:.3f} | {d['micro']['f1']:.3f} | {d['f1_ci95'][0]:.3f}–{d['f1_ci95'][1]:.3f} | "
          f"{d['fp_per_image']:.2f} | {pc['pothole']['f1']:.3f} | {pc['crack']['f1']:.3f} | {pc['manhole']['f1']:.3f} |")

plt.figure(figsize=(8, 4))
for n in ('E1', 'E2', 'E3'):
    L = json.load(open(f'{BACKUP}/{n}/run.json'))['losses']
    plt.plot([sum(L[max(0, i-9):i+1]) / len(L[max(0, i-9):i+1]) for i in range(len(L))], label=n)
plt.xlabel('bước'); plt.ylabel('loss (trung bình trượt 10)'); plt.legend(); plt.grid(alpha=.3)
os.makedirs('report/figures', exist_ok=True); plt.savefig('report/figures/loss.png', dpi=120, bbox_inches='tight'); plt.show()

BEST = max(('E1', 'E2', 'E3'), key=lambda n: EV[n]['micro']['f1'])
rows = per_image(S['test'], read_jsonl(f'{BACKUP}/{BEST}/pred_test.jsonl'))
good = sorted([r for r in rows if r[2] and not r[3] and not r[4]], key=lambda r: -r[2])[:3]
bad = sorted(rows, key=lambda r: -(r[3] + r[4]))[:3]
figs = []
for tag, rs in (('dung', good), ('sai', bad)):
    for i, (e, raw, tp, fp, fn) in enumerate(rs):
        pth = f'report/figures/{BEST}_{tag}_{i}.jpg'
        draw(e, raw, pth, f'{BEST} {e["image_id"]} TP={tp} FP={fp} FN={fn}'); figs.append(pth)
print('setup tốt nhất:', BEST); show(figs)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

**E4 — biến `data_size`** (10% vs 100% ảnh train), mọi thứ khác giữ như setup tốt nhất. Giá trị 1.0 chính là setup tốt nhất nên chấm lại predictions của nó dưới tên E4.

In [ ]:
cfg4 = {**CFG[BEST.lower()], 'name': 'E4_data_0.1'}
R['E4_0.1'], EV['E4_DATA_0.1'] = experiment(cfg4, f'{BACKUP}/E4_data_0.1', 'report/evidence/e4_data_0.1.json', train_fraction=0.1)
run_best = json.load(open(f'{BACKUP}/{BEST}/run.json'))
EV['E4_DATA_1.0'] = score(f'{BACKUP}/{BEST}/pred_test.jsonl', 'E4_data_1.0', 'report/evidence/e4_data_1.0.json',
                          {**run_meta(run_best), 'train_fraction': 1.0, 'base_setup': BEST})

**E5 (tuỳ chọn) — GRPO với reward IoU**, khởi tạo từ adapter tốt nhất. Hàm `compute_reward` có test trong `test_lab.py`.

In [ ]:
!python -m pytest -q test_lab.py

In [ ]:
run5 = grpo(f'{BACKUP}/{BEST}', f'{BACKUP}/E5', CFG['e5']['grpo'])
json.dump(run5['log'], open('report/e5_grpo_log.json', 'w'), indent=1)
lg = run5['log']
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([l['reward_mean'] for l in lg]); ax[0].set_title('reward trung bình / bước')
ax[1].plot([l['mean_n_pred'] for l in lg], label='số box dự đoán TB'); ax[1].plot([l['empty'] for l in lg], label='số output "[]"')
ax[1].plot([l['giant'] for l in lg], label='số output có box >25% ảnh'); ax[1].legend(); ax[1].set_title('dấu hiệu reward hacking')
plt.savefig('report/figures/e5_grpo.png', dpi=120, bbox_inches='tight'); plt.show()
if not os.path.exists(f'{BACKUP}/E5/pred_test.jsonl') or len(read_jsonl(f'{BACKUP}/E5/pred_test.jsonl')) < 311:
    model, processor = load_trained(f'{BACKUP}/E5')
    predict(model, processor, S['test'], f'{BACKUP}/E5/pred_test.jsonl', max_new_tokens=512); del model; cleanup()
EV['E5'] = score(f'{BACKUP}/E5/pred_test.jsonl', 'E5', 'report/evidence/e5.json', {**run_meta(run5), 'init': BEST})

## Xuất kết quả về máy
Evidence/ảnh nằm trên VM Colab. Ô dưới gói `report/` + predictions + run.json thành base64 trong output; lưu notebook (Ctrl+S) là máy local có dữ liệu để commit.

In [ ]:
import base64, io, tarfile, glob
buf = io.BytesIO()
with tarfile.open(fileobj=buf, mode='w:gz') as t:
    t.add('report'); t.add('data/splits')
    for f in glob.glob(f'{BACKUP}/*/pred_test.jsonl') + glob.glob(f'{BACKUP}/*/run.json'):
        t.add(f, arcname='work/' + f[len(BACKUP) + 1:])
open(f'{BACKUP}/export.tar.gz', 'wb').write(buf.getvalue())
print('EXPORT_B64:' + base64.b64encode(buf.getvalue()).decode())